# Deploy a Qwen3.6-27B on SageMaker Endpoint using AWS Python API (boto3)



### [Qwen3.6](https://huggingface.co/collections/Qwen/qwen36) is a series of models built on direct feedback from the community, Qwen3.6 prioritizes stability and real-world utility, offering developers a more intuitive, responsive, and genuinely productive coding experience..

#### Qwen3.6 Highlights

**Qwen3.6 features the following enhancement**:

This release delivers substantial upgrades, particularly in

- *Agentic Coding*: the model now handles frontend workflows and repository-level reasoning with greater fluency and precision.

- *Thinking Preservation*: we've introduced a new option to retain reasoning context from historical messages, streamlining iterative development and reducing overhead.

Please refer to official [blog](https://qwen.ai/blog?id=qwen3.6-27b) for more details


In [ ]:
%pip install --upgrade --quiet --no-warn-conflicts boto3

In [ ]:
import time
import re
import json
import boto3
from IPython.display import display, Markdown, clear_output

boto_session = boto3.Session()
region = boto_session.region_name

sm = boto3.client("sagemaker")  # client to intreract with SageMaker
sm_runtime = boto3.client("sagemaker-runtime")  # client to intreract with SageMaker Endpoints

In [ ]:
#
# Helper functions to remove dependency on SageMaker Python SDK
#
def get_sagemaker_role():
    sts = boto3.client('sts')
    response = sts.get_caller_identity()
    assumed_role = response['Arn']
    role = re.sub(r"^(.+)sts::(\d+):assumed-role/(.+?)/.*$", r"\1iam::\2:role/\3", assumed_role)
    return role


def wait_for_endpoint(endpoint_name: str, sleep_time: int=60):
    ind = "."
    progress = f"Waiting for '{endpoint_name}': "
    print(progress)
    
    status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
    
    while status == "Creating":
        time.sleep(sleep_time)
        
        status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
        
        clear_output(wait=True)
        progress += ind
        print(progress)
  
    print(f"Endpoint: '{endpoint_name}', Status: '{status}'")


def wait_for_ic(ic_name: str, sleep_time: int=60):
    ind = "."
    progress = f"Waiting for '{ic_name}': "
    print(progress)
    
    status = sm.describe_inference_component(InferenceComponentName=ic_name)["InferenceComponentStatus"]
    
    while status == "Creating":
        time.sleep(sleep_time)
        
        status = sm.describe_inference_component(InferenceComponentName=ic_name)["InferenceComponentStatus"]
        
        clear_output(wait=True)
        progress += ind
        print(progress)
  
    print(f"IC: '{ic_name}', Status: '{status}'")

In [ ]:
#
# Overwrite with your role ARN if you are running this notebook outside of SageMaker Studio
#
role = None

if role == None:
    role = get_sagemaker_role()
print(role)

## Container

In [ ]:
instance = {"type": "ml.g6e.12xlarge", "num_gpu": 4}
model_id = "Qwen/Qwen3.6-27B"
model_name = f"model-{time.strftime('%y%m%d-%H%M%S')}"
endpoint_name = model_name
endpoint_config_name = model_name
timeout = 900
variant_name = "v1"

##### Amazon SageMaker AI makes model deployment process very easy. The deployment steps are exactly the same regardless of model framework container you choose. Below we show you 3 options:
1. DLC vLLM (0.19.1)
2. DLC SGLang (0.5.9)
3. SageMaker LMI (v22)

#### Please chose one option and run only the cell for your container of choice

#### We will be using native Qwen3.5 multi-token predictions (MTP). The net effect is higher throughput (more tokens per second) with no degradation in output quality, since the verification step ensures the final output matches what standard autoregressive decoding would produce. It's essentially "free" speculation baked into the model itself, removing the overhead of training and serving a separate smaller draft model.

### Option 1: vLLM

In [ ]:
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/vllm:0.19.0-gpu-py312-cu129-ubuntu22.04-sagemaker"

spec_config = {
    "method": "qwen3_next_mtp",
    "num_speculative_tokens": 2
}

common_env = {
    "HF_MODEL_ID": model_id,
}

vllm_env = {
    "SM_VLLM_MODEL": model_id,
    "SM_VLLM_TENSOR_PARALLEL_SIZE": json.dumps(instance["num_gpu"]),
    "SM_VLLM_MAX_MODEL_LEN": "32768",
    "SM_VLLM_ENABLE_AUTO_TOOL_CHOICE": "true",
    "SM_VLLM_TOOL_CALL_PARSER": "qwen3_coder",
    "SM_VLLM_REASONING_PARSER": "qwen3",
    "SM_VLLM_SPECULATIVE_CONFIG": json.dumps(spec_config)
}
env = common_env | vllm_env

### Option 2: SGLang

In [ ]:
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/sglang:0.5.9-gpu-py312-cu129-ubuntu24.04-sagemaker"

common_env = {
    "SGLANG_DISABLE_CUDNN_CHECK": "1",
    "SM_SGLANG_ATTENTION_BACKEND": "triton",
    "SM_SGLANG_MEM_FRACTION_STATIC": "0.8",
    "SM_SGLANG_FP8_GEMM_BACKEND": "triton",
    "SM_SGLANG_MOE_RUNNER_BACKEND": "triton",
    "SM_SGLANG_MODEL_PATH": model_id
}
sgl_env = {
    "SM_SGLANG_TENSOR_PARALLEL_SIZE": json.dumps(instance["num_gpu"]),
    "SM_SGLANG_CONTEXT_LENGTH": "32768",
    "SM_SGLANG_TOOL_CALL_PARSER": "qwen3_coder",
    "SM_SGLANG_REASONING_PARSER": "qwen3",
    "SM_SGLANG_SPECULATIVE_ALGORITHM": "EAGLE",
    "SM_SGLANG_SPECULATIVE_NUM_STEPS": "3",
    "SM_SGLANG_SPECULATIVE_EAGLE_TOPK": "1",
    "SM_SGLANG_SPECULATIVE_NUM_DRAFT_TOKENS": "4",
    "SM_SGLANG_ENABLE_FLASHINFER_ALLREDUCE_FUSION": "true",
}
env = common_env | sgl_env

### Option 3: LMI

In [ ]:
CONTAINER_VERSION = "0.36.0-lmi22.0.0-cu129"
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/djl-inference:{CONTAINER_VERSION}"

spec_config = {
    "method": "mtp",
    "num_speculative_tokens": 3
}
common_env = {
    "HF_MODEL_ID": model_id,
}
lmi_env = {
    "SERVING_FAIL_FAST": "true",
    "OPTION_ASYNC_MODE": "true",
    "OPTION_TENSOR_PARALLEL_DEGREE": json.dumps(instance["num_gpu"]),
    "OPTION_MAX_MODEL_LEN": "32768",
    "OPTION_ENABLE_AUTO_TOOL_CHOICE": "true",
    "OPTION_TOOL_CALL_PARSER": "qwen3_coder",
    "OPTION_REASONING_PARSER": "qwen3",
    "OPTION_SPECULATIVE_CONFIG": json.dumps(spec_config)
}
env = common_env | lmi_env

## Deployment

In [ ]:
_ = sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": inference_image,
        "Environment": env,
    },
)

In [ ]:
_ = sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            "VariantName": variant_name,
            "ModelName": model_name,
            "InstanceType": instance["type"],
            "InitialInstanceCount": 1,
            "ContainerStartupHealthCheckTimeoutInSeconds": timeout,
        },
    ],
)

_ = sm.create_endpoint(EndpointName=endpoint_name, 
                       EndpointConfigName=endpoint_config_name)

_ = wait_for_endpoint(endpoint_name)

---
## Inference examples

### Best Practices

We recommend using the following set of sampling parameters for generation

- Thinking mode for general tasks: temperature=1.0, top_p=0.95, top_k=20, min_p=0.0, presence_penalty=0.0, repetition_penalty=1.0

- Thinking mode for precise coding tasks (e.g. WebDev): temperature=0.6, top_p=0.95, top_k=20, min_p=0.0, presence_penalty=0.0, repetition_penalty=1.0

- Instruct (or non-thinking) mode: temperature=0.7, top_p=0.80, top_k=20, min_p=0.0, presence_penalty=1.5, repetition_penalty=1.0

Please note that the support for sampling parameters varies according to inference frameworks.

Qwen3.6 models operate in thinking mode by default, generating thinking content signified by <think>\n...</think>\n\n before producing the final responses. To disable thinking content and obtain direct response, refer to the examples here.

---
#### Text inference

In [68]:
payload={
    "messages": [
        {"role": "user", "content": "Name popular places to visit in London?"}
    ],
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()

print(f"✅ Response time: {end_time-start_time:.2f}s\n")
display(Markdown(response["choices"][0]["message"]["content"]))

usage = response["usage"] 
print(f'-----------------------\n{usage}')

✅ Response time: 25.35s





Here’s a curated list of London’s most popular and highly recommended attractions, grouped by category for easier planning:

### 👑 Royal & Historic
- **Tower of London**: Historic fortress, home to the Crown Jewels, and once a palace, prison, and zoo. Book ahead for skip-the-line access.
- **Buckingham Palace**: Official residence of the King. Watch the Changing of the Guard (schedule varies) and visit the State Rooms in summer.
- **Westminster Abbey & Houses of Parliament/Big Ben**: Gothic masterpiece and iconic clock tower. Guided tours of the Palace of Westminster require advance booking.

### 🏛️ Museums & Galleries (Mostly Free)
- **British Museum**: One of the world’s greatest collections of human history, including the Rosetta Stone and Egyptian mummies.
- **Victoria & Albert Museum (V&A)**: Premier museum of art, design, and performance, with everything from fashion to architectural models.
- **Tate Modern**: Leading contemporary art gallery housed in a converted Thames-side power station. The free Turbine Hall installations are a highlight.
- **Natural History Museum**: Stunning Romanesque building with dinosaur skeletons, planetarium, and family-friendly exhibits.

### 🌆 Iconic Landmarks & Views
- **London Eye**: Giant observation wheel with 360° views of the city and Thames. Book online to avoid long queues.
- **Tower Bridge**: Victorian engineering marvel. Walk the high-level glass walkways or explore the Victorian Engine House.
- **Sky Garden**: Free-to-enter sky-high public space in 20 Fenchurch Street with lush terraces and city views. **Book tickets in advance** (often released 1–2 weeks ahead).
- **The Shard**: London’s tallest building with a paid viewing gallery on floors 68–72.

### 🍽️ Markets & Cultural Hubs
- **Borough Market**: Historic food market (since 1014) with artisan producers, street food, and fresh produce. Best on weekday mornings.
- **Covent Garden**: Street performers, boutique shops, restaurants, and the Royal Opera House. Great for people-watching and shopping.
- **Camden Market**: Alternative culture, vintage clothing, global street food, and live music along Regent’s Canal.

### 🌳 Parks & Green Spaces
- **Hyde Park & The Serpentine**: London’s largest Royal Park. Rent a boat, visit Kensington Palace, or relax by the lake.
- **Kew Gardens**: UNESCO-listed botanical gardens with exotic glasshouses, tree houses, and seasonal flower displays. Located in southwest London.

### 💡 Quick Tips:
- **Free entry** applies to most major museums and galleries (donations welcome).
- **Book ahead** for popular paid attractions (Tower of London, London Eye, Sky Garden, Palace of Westminster, etc.).
- **Transport**: Use contactless bank cards, Apple/Google Pay, or an Oyster card for buses and the Tube. Walk or cycle where possible.
- **Consider a pass**: If planning 3+ paid attractions, a London Pass or London Experience Day may save money.

Let me know your interests (history, family-friendly, food, budget, 2-day vs. week-long trip, etc.) and I can tailor a customized itinerary!

-----------------------
{'prompt_tokens': 18, 'total_tokens': 1778, 'completion_tokens': 1760, 'prompt_tokens_details': None}


---
#### Video Understanding



<video src="https://qianwen-res.oss-accelerate.aliyuncs.com/Qwen3.5/demo/video/N1cdUjctpG8.mp4">

In [69]:
payload={
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "video_url",
                    "video_url": {
                        "url": "https://qianwen-res.oss-accelerate.aliyuncs.com/Qwen3.5/demo/video/N1cdUjctpG8.mp4"
                    }
                },
                {
                    "type": "text",
                    "text": "How many porcelain jars were discovered in the niches located in the primary chamber of the tomb?"
                }
            ]
        }
    ],
    "extra_body": {"top_k": 20, "mm_processor_kwargs": {"fps": 2, "do_sample_frames": "True"}}
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()

print(f"✅ Response time: {end_time-start_time:.2f}s\n")
display(Markdown(response["choices"][0]["message"]["content"]))

usage = response["usage"] 
print(f'-----------------------\n{usage}')

✅ Response time: 180.72s





Based on the visual evidence in the video, specifically the image showing the interior of the tomb (around 00:10), there appear to be **9** porcelain jars.

There is a row of 8 jars lined up against the brick wall (alternating between larger jar-shaped vessels and smaller bowl-shaped vessels) and one additional jar sitting on the floor to the left. This aligns with reports about this specific archaeological find, which noted that 9 porcelain jars were discovered in the niches of the main chamber.

-----------------------
{'prompt_tokens': 11692, 'total_tokens': 14781, 'completion_tokens': 3089, 'prompt_tokens_details': None}


---
#### Text generation with reasoning output

In [70]:
payload={
    "messages": [
        {"role": "user", "content": "What is the derivative of x^3 * ln(x)?"}
    ],
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()

print(f"✅ Response time: {end_time-start_time:.2f}s\n")
#print(json.dumps(response, indent=2, default=str))
if response["choices"][0]["message"]["reasoning"]:
    display(Markdown('### Reasoning:\n---'))
    display(Markdown(response["choices"][0]["message"]["reasoning"]))
    #display(Markdown('---'))
display(Markdown('### Content:\n---'))
display(Markdown(response["choices"][0]["message"]["content"]))
display(Markdown('---'))

usage = response["usage"] 
print(f'-----------------------\n{usage}')

✅ Response time: 14.95s



### Reasoning:
---

Here's a thinking process:

1.  **Understand the User's Request:** The user is asking for the derivative of the function $f(x) = x^3 \ln(x)$.

2.  **Identify the Mathematical Operation:** I need to compute $\frac{d}{dx}[x^3 \ln(x)]$.

3.  **Recognize the Structure:** The function is a product of two functions of $x$: $u(x) = x^3$ and $v(x) = \ln(x)$. This means I should use the **Product Rule**.

4.  **Recall the Product Rule:** 
   $\frac{d}{dx}[u(x)v(x)] = u'(x)v(x) + u(x)v'(x)$

5.  **Compute Individual Derivatives:**
   - $u(x) = x^3 \Rightarrow u'(x) = 3x^2$ (using Power Rule)
   - $v(x) = \ln(x) \Rightarrow v'(x) = \frac{1}{x}$ (standard derivative)

6.  **Apply the Product Rule:**
   $\frac{d}{dx}[x^3 \ln(x)] = (3x^2)(\ln(x)) + (x^3)\left(\frac{1}{x}\right)$

7.  **Simplify the Expression:**
   - First term: $3x^2 \ln(x)$
   - Second term: $x^3 \cdot \frac{1}{x} = x^2$
   - Combine: $3x^2 \ln(x) + x^2$
   - Factor if desired: $x^2(3\ln(x) + 1)$

8.  **Verify/Double Check:**
   - Let's differentiate again mentally: $f(x) = x^3 \ln x$
   - $f'(x) = 3x^2 \ln x + x^3 \cdot (1/x) = 3x^2 \ln x + x^2$. Correct.
   - Domain: $x > 0$ for $\ln(x)$ to be defined, so the derivative is valid for $x > 0$.
   - The steps are solid.

9.  **Formulate the Response:**
   - State the rule used (Product Rule).
   - Show step-by-step differentiation.
   - Provide the simplified final answer.
   - Optionally mention the factored form.
   - Keep it clear and concise.

   Draft:
   To find the derivative of $f(x) = x^3 \ln(x)$, we use the **product rule**: $\frac{d}{dx}[u(x)v(x)] = u'(x)v(x) + u(x)v'(x)$.
   Let $u(x) = x^3$ and $v(x) = \ln(x)$.
   Then:
   - $u'(x) = 3x^2$
   - $v'(x) = \frac{1}{x}$
   Applying the product rule:
   $f'(x) = (3x^2)(\ln(x)) + (x^3)\left(\frac{1}{x}\right)$
   Simplify:
   $f'(x) = 3x^2 \ln(x) + x^2$
   This can also be factored as:
   $f'(x) = x^2(3\ln(x) + 1)$
   (Valid for $x > 0$)

10. **Final Review:** The response matches the draft, is mathematically correct, and directly answers the question. No extra fluff. Ready.✅
   Output matches the response.✅
   Proceed. 
   [Self-Correction/Refinement during thought]: 
   - Check if any special cases or domain restrictions need emphasis? The problem is straightforward calculus. Mentioning $x>0$ is good practice but not strictly required unless asked. I'll keep it brief.
   - The steps are clear. I'll present it neatly.✅
   Output generation. 
   [Output Generation] -> *Proceeds*
   *(Self-Correction/Verification during generation prep)*: 
   Derivative of x^3 is 3x^2. Derivative of ln(x) is 1/x. Product rule: (3x^2)(ln x) + (x^3)(1/x) = 3x^2 ln x + x^2. Correct. Factored: x^2(3 ln x + 1). Correct. All good.✅
   Output matches the final response. 
   [Done.] 
   *Output Generation* (matches the final refined version)
   "To find the derivative of $f(x) = x^3 \ln(x)$, use the **product rule**: ..." etc.✅
   [Final Check of the Prompt]: "What is the derivative of x^3 * ln(x)?" -> Direct, expects formula/steps.
   All set. 
   *Proceeds to output* ✅


### Content:
---



To find the derivative of $f(x) = x^3 \ln(x)$, use the **product rule**:
$$\frac{d}{dx}[u(x)v(x)] = u'(x)v(x) + u(x)v'(x)$$

Let:
* $u(x) = x^3 \quad \Rightarrow \quad u'(x) = 3x^2$
* $v(x) = \ln(x) \quad \Rightarrow \quad v'(x) = \frac{1}{x}$

Apply the rule:
$$f'(x) = (3x^2)(\ln(x)) + (x^3)\left(\frac{1}{x}\right)$$

Simplify:
$$f'(x) = 3x^2 \ln(x) + x^2$$

This can also be factored as:
$$\boxed{f'(x) = x^2(3\ln(x) + 1)}$$

*(Note: This derivative is valid for $x > 0$, since $\ln(x)$ is only defined for positive $x$.)*

---

-----------------------
{'prompt_tokens': 22, 'total_tokens': 1368, 'completion_tokens': 1346, 'prompt_tokens_details': None}


---
#### Streaming

In [77]:
#
# Helper function for streamin invocation
#
import io
import json
import time
import boto3
from IPython.display import clear_output

class LineIterator:
    def __init__(self, stream):
        self.byte_iterator = iter(stream)
        self.buffer = io.BytesIO()
        self.read_pos = 0

    def __iter__(self):
        return self

    def __next__(self):
        while True:
            self.buffer.seek(self.read_pos)
            line = self.buffer.readline()
            if line and line[-1] == ord("\n"):
                self.read_pos += len(line)
                return line[:-1]
            try:
                chunk = next(self.byte_iterator)
            except StopIteration:
                if self.read_pos < self.buffer.getbuffer().nbytes:
                    continue
                raise
            if "PayloadPart" not in chunk:
                print("Unknown event type:" + chunk)
                continue
            self.buffer.seek(0, io.SEEK_END)
            self.buffer.write(chunk["PayloadPart"]["Bytes"])

def stream_response(endpoint_name, inputs, max_tokens=8189, temperature=0.7, top_p=0.9):    
    body = {
      "messages": [
            {"role": "user", "content": [{"type": "text", "text": inputs}]}
        ],
        "max_tokens": max_tokens,
        "temperature": temperature,
        "top_p": top_p,
        "stream": True,
        "chat_template_kwargs": {"enable_thinking": "true"}
    }

    resp = sm_runtime.invoke_endpoint_with_response_stream(
        EndpointName = endpoint_name,
        Body = json.dumps(body),
        ContentType = "application/json",
    )

    event_stream = resp["Body"]
    start_json = b"{"
    full_response = ""
    start_time = time.time()
    token_count = 0

    for line in LineIterator(event_stream):
        if line != b"" and start_json in line:
            data = json.loads(line[line.find(start_json):].decode("utf-8"))
            delta = data['choices'][0]['delta']
            if "reasoning" in delta:
                token_text = delta.get('reasoning', ' ')
            elif "content" in delta:
                token_text = delta.get('content', ' ')          
            else:
                token_text = ' '
            full_response += token_text
            token_count += 1

            # Calculate tokens per second
            elapsed_time = time.time() - start_time
            tps = token_count / elapsed_time if elapsed_time > 0 else 0

            # Clear the output and reprint everything
            clear_output(wait=True)
            display(Markdown(full_response))

    print(f"\nTokens per Second: {tps:.2f}\n")

In [78]:
#inputs = "What is greater 9.11 or 9.8?"
inputs = "Solve this problem step by step: What is 15% of 240?"
output = stream_response(endpoint_name, inputs, max_tokens=8000)

Here's a thinking process:

1.  **Understand the User's Request:** The user wants to find 15% of 240, and they want the solution explained step by step.

2.  **Identify the Core Mathematical Operation:** Finding a percentage of a number involves converting the percentage to a decimal or fraction and then multiplying it by the given number.
   - Formula: (Percentage / 100) × Number
   - Here: (15 / 100) × 240

3.  **Break Down the Steps:**
   - Step 1: Convert the percentage to a decimal or fraction.
   - Step 2: Multiply the decimal/fraction by the number.
   - Step 3: Calculate the result.
   - Step 4: State the final answer clearly.

4.  **Execute the Steps (Mental Calculation):**
   - Step 1: 15% = 15/100 = 0.15
   - Step 2: 0.15 × 240
   - Step 3: Calculation:
     - 0.15 × 240 = (15 × 240) / 100
     - 15 × 240 = 15 × 24 × 10 = 360 × 10 = 3600
     - 3600 / 100 = 36
     - Alternatively: 10% of 240 = 24, 5% of 240 = 12, so 15% = 24 + 12 = 36.
   - Step 4: The answer is 36.

5.  **Structure the Response:** Present the steps clearly as requested.
   - Start with a clear statement of the problem.
   - Show the conversion of percentage to decimal/fraction.
   - Show the multiplication.
   - Provide an alternative method (like breaking into 10% + 5%) for clarity/verification.
   - State the final answer.

   Draft:
   To find 15% of 240, follow these steps:
   Step 1: Convert the percentage to a decimal.
   15% = 15 ÷ 100 = 0.15
   Step 2: Multiply the decimal by the number.
   0.15 × 240 = ?
   Step 3: Perform the multiplication.
   You can calculate this by breaking it down:
   0.15 × 240 = (15 × 240) ÷ 100
   15 × 240 = 3,600
   3,600 ÷ 100 = 36
   (Alternatively, 10% of 240 is 24, and 5% is half of that, which is 12. So 15% = 24 + 12 = 36.)
   Step 4: State the final answer.
   15% of 240 is 36.

6.  **Self-Correction/Verification:**
   - Check calculation: 0.15 * 240 = 36. Correct.
   - Check alternative method: 10% = 24, 5% = 12, sum = 36. Correct.
   - Steps are logical and easy to follow.
   - Matches user request exactly.

   The response is ready. I'll output it exactly as structured.✅


Here's how to solve it step by step:

**Step 1: Convert the percentage to a decimal**
To convert a percentage to a decimal, divide by 100:
`15% = 15 ÷ 100 = 0.15`

**Step 2: Multiply the decimal by the number**
Now, multiply 0.15 by 240:
`0.15 × 240 = ?`

**Step 3: Calculate the product**
You can do this directly or break it down:
- `0.15 × 240 = (15 × 240) ÷ 100`
- `15 × 240 = 3,600`
- `3,600 ÷ 100 = 36`

*(Quick check method: 10% of 240 is 24. 5% is half of that, which is 12. So 15% = 24 + 12 = 36.)*

**Step 4: State the final answer**
`15% of 240 = 36`

**Answer:** 36


Tokens per Second: 32.08



---
#### Extracting text from image

![image](https://ofasys-multimodal-wlcb-3-toshanghai.oss-accelerate.aliyuncs.com/wpf272043/keepme/image/receipt.png)

In [79]:
payload = {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "https://ofasys-multimodal-wlcb-3-toshanghai.oss-accelerate.aliyuncs.com/wpf272043/keepme/image/receipt.png"
                    }
                },
                {
                    "type": "text",
                    "text": "Read all the text in the image."
                }
            ]
        }
    ],
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()

print(f"✅ Response time: {end_time-start_time:.2f}s\n")
display(Markdown(response["choices"][0]["message"]["content"]))

usage = response["usage"] 
print(f'-----------------------\n{usage}')

✅ Response time: 3.41s





Auntie Anne's
CINNAMON SUGAR
1 x 17.000 17,000
SUB TOTAL 17,000
GRAND TOTAL 17,000
CASH IDR 20,000
CHANGE DUE 3,000

-----------------------
{'prompt_tokens': 249, 'total_tokens': 424, 'completion_tokens': 175, 'prompt_tokens_details': None}


---
### Performance improvement by using speculative decoding

![SD vs no-SD Performance](sd_vs_nosd.png)

## Key Takeaways

| Metric | Concurrency | SD | no-SD | SD Improvement |
|---|---|---|---|---|
| **Latency** | 1 | 2,894 ms | 4,885 ms | **−41%** |
| **Latency** | 10 | 6,427 ms | 8,775 ms | **−27%** |
| **Throughput** | 1 | 137.2 tokens/sec | 81.2 tokens/sec | **+69%** |
| **Throughput** | 10 | 528.0 tokens/sec | 383.6 tokens/sec | **+38%** |

### Summary

- **SD consistently outperforms no-SD** across both latency and throughput at all concurrency levels.
- **Largest latency gain at concurrency 1**: SD reduces average request latency by **41%** (from 4,885 ms to 2,894 ms).
- **Largest throughput gain at concurrency 1**: SD delivers **69% more tokens/sec** (137 vs 81).
- At **concurrency 10**, SD still provides a **27% latency reduction** and **38% throughput increase**.
- Scaling from concurrency 1 → 10 increases throughput ~3.9× for SD and ~4.7× for no-SD, while latency roughly doubles for both.

## Cleanup

In [80]:
_ = sm.delete_endpoint(EndpointName=endpoint_name)
_ = sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
_ = sm.delete_model(ModelName=model_name)